# Demos: Lecture 01

In [ ]:
import pennylane as qp
from pennylane import numpy as np
import matplotlib.pyplot as plt

## Demo 1: $T_1$ characterization

### Version 1: MC sampling

In [ ]:
τ = 60
times = np.arange(0, 300, 5)

@qp.set_shots(1)
@qp.qnode(qp.device('default.qubit', wires=1))
def spontaneous_relaxtion(t):
    qp.PauliX(0)

    relaxation_prob = 1 - np.exp(-t/τ)
    if np.random.uniform() < relaxation_prob:
        qp.PauliX(0)
    
    return qp.sample()

In [ ]:
N_shots = 1000
survival_probs = np.zeros(len(times))

for idx, t in enumerate(times):
    measurement_samples = np.array([spontaneous_relaxtion(t) for _ in range(N_shots)])
    survival_probs[idx] = np.mean(measurement_samples)

In [ ]:
plt.scatter(times, survival_probs)
plt.xlabel("Time (ns)")
plt.ylabel("Survival probability")

In [ ]:
from scipy.optimize import curve_fit

def exp_func(t, τ, a, b):
    return a * np.exp(-t / τ) + b

popt, pcov = curve_fit(exp_func, times, survival_probs)

In [ ]:
popt

In [ ]:
plt.scatter(times, survival_probs, label="Data")
plt.plot(times, exp_func(times, *popt), c='tab:orange', linestyle="--", label="Fit")
plt.legend()
plt.xlabel("Time (ns)")
plt.ylabel("Survival probability")

### Version 2: amplitude damping channel

In [ ]:
@qp.qnode(qp.device('default.mixed', wires=1))
def spontaneous_relaxtion_ampdamp(t):
    qp.PauliX(0)
    
    relaxation_prob = 1 - np.exp(-t/τ)
    qp.AmplitudeDamping(relaxation_prob, 0)

    return qp.probs()

In [ ]:
survival_probs_ampdamp = np.array([spontaneous_relaxtion_ampdamp(t)[1] for t in times])

In [ ]:
plt.scatter(times, survival_probs, label="Data")
plt.plot(times, exp_func(times, *popt), c='tab:orange', linestyle="--", label="Fit")
plt.plot(times, survival_probs_ampdamp, c='tab:green', label="Analytic")
plt.xlabel("Time (ns)")
plt.ylabel("Survival probability")
plt.legend()

## Demo 2: $T_2$ and $T_2^*$ characterization (Hahn-echo and Ramsey experiments)

### Version 1: Hahn-echo with MC simulation

In [ ]:
τ = 50

ω = 0.01 # Intrinsic qubit frequency

times = np.arange(0, 200, 2)

@qp.set_shots(1)
@qp.qnode(qp.device('default.qubit', wires=1))
def hahn_echo(t):
    phase_flip_prob = 1 - np.exp(-t/τ)
    
    qp.SX(0)

    qp.RZ(2 * np.pi * ω * t, 0)
            
    qp.X(0)
    
    qp.RZ(2 * np.pi * ω * t, 0)

    # Try yourself: instead of phase flip, model noise as a small,
    # random additional phase (see alternative Kraus characterization  
    # in supplementary exercises)    
    if np.random.uniform() < phase_flip_prob:
        qp.PauliZ(0)
    
    qp.SX(0)
    
    return qp.sample()

In [ ]:
N_shots = 100
survival_probs = np.zeros(len(times))

for idx, t in enumerate(times):
    measurement_samples = np.array([hahn_echo(t) for _ in range(N_shots)])
    survival_probs[idx] = 1 - np.mean(measurement_samples)

In [ ]:
plt.scatter(times, survival_probs)
plt.xlabel("Time (ns)")
plt.ylabel("Survival probability")

In [ ]:
popt, pcov = curve_fit(exp_func, times, survival_probs)

### Version 2: Ramsey interferometry with MC simulation

Try implementing this yourself based on the explanation [here](https://qiskit-community.github.io/qiskit-experiments/manuals/characterization/t2ramsey.html).

In [ ]:
@qp.set_shots(1)
@qp.qnode(qp.device('default.qubit', wires=1))
def ramsey_interferometry(t):
    # Your code here
    return qp.sample()